# AlphaPulldown on Colab — GPU-only bait/prey screening
Updated 4 October 2026. Uses **AlphaPulldown 2.9.1**, its AlphaFold2 dependency extra, and AlphaFold-Multimer v3 weights. Requires precomputed AlphaPulldown monomer objects (`.pkl` or `.pkl.xz`), not raw A3M alignments. This notebook does not generate MSAs.

1. Start a fresh Colab GPU runtime (Runtime → Change runtime type; A100 if available).
2. Run the bootstrap cell alone; CondaColab restarts the kernel. Continue with the next cell after reconnecting.
3. Edit the configuration, then run the remaining cells in order. Initially run one pair.
4. Reconnect/reinstall after a runtime reset and use the same Drive output folder to resume. Completed jobs and partial prediction files are saved directly to Drive.

This adaptation has been checked statically, but has not been executed on a Colab GPU. Older feature pickles may need regeneration with 2.9.1; the validation cell checks deserialization and basic contents before prediction.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'], check=True)
import condacolab
condacolab.install()



📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


## 1. Mount Drive and configure the screen
Create `baits.txt` with one bait name per line, and `preys.txt` with one candidate name per line. Names must match feature filenames, excluding `.pkl` or `.pkl.xz`. Simple whole-protein identifiers only: no residue/copy syntax here. Blank lines and lines beginning with `#` are ignored. Your original `syd.txt` is retained as the default candidate list.

Use a new `RUN_NAME` when changing model settings, protein sequences/features, or software versions. Input changes within an existing run are refused to avoid mixing results.


In [15]:
from google.colab import drive
from pathlib import Path
import os, json, subprocess, hashlib, shutil, sys, time

drive.mount('/content/drive')
AP_VERSION = '2.9.1'
ENV_PREFIX = Path('/content/ap_env')
FEATURE_DIR = Path('/content/drive/MyDrive/colab_ec/Escherichia_coli/features')
BAIT_LIST = Path('/content/drive/MyDrive/colab_ec/mrcB_799-844.fasta')
PREY_LIST = Path('/content/drive/MyDrive/colab_ec/K12_periplasmic_domain_only_accessions.fasta')
OUTPUT_ROOT = Path('/content/drive/MyDrive/colab_ec/out')
RUN_NAME = '1b_K12'
PARAM_ROOT = Path('/content/par')  # Files are in PARAM_ROOT / 'params'.

NUM_RECYCLES = 3
PREDICTIONS_PER_MODEL = 2  # Five model parameter sets × two predictions = ten per pair.
MODEL_NAMES = [f'model_{i}_multimer_v3' for i in range(1, 6)]
PAIR_MSA = True
SKIP_TEMPLATES = False
BASE_SEED = 42
START_JOB = 1  # 1-based inclusive, in the saved jobs.csv manifest.
END_JOB = 1    # Start with one pair; set None to run to the end.
CONTINUE_ON_ERROR = False

RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
# Inherit Colab's CUDA library setup; let JAX allocate only what it needs.
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
CONDA = shutil.which('conda')
if not CONDA:
    raise RuntimeError('Run the CondaColab bootstrap, wait for restart, then resume here.')

def ap_run(args, **kwargs):
    return subprocess.run([CONDA, 'run', '--no-capture-output', '-p', str(ENV_PREFIX), *args],
                          env=RUNTIME_ENV, check=True, **kwargs)
print('Run directory:', RUN_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run directory: /content/drive/MyDrive/colab_ec/out/1b_K12


## 2. Install the pinned AF2 environment
Uses a separate Python 3.10 environment, so Colab's own notebook packages are not replaced. The AF2 extra selects the release's JAX/CUDA, OpenMM and PDBFixer dependencies. No separate JAX upgrade is needed. Installation stops on dependency errors.

Setuptools is constrained below 82 because PDBFixer imports the removed `pkg_resources` module.


In [ ]:
if not (ENV_PREFIX / 'bin/python').exists():
    subprocess.run([CONDA, 'create', '-y', '-p', str(ENV_PREFIX), '-c', 'conda-forge',
                    'python=3.10', 'pip'], check=True)
ap_run(['python', '-m', 'pip', 'install', '-v', f'alphapulldown[alphafold2]=={AP_VERSION}', 'setuptools<82'])
ap_run(['python', '-m', 'pip', 'check', '-v'])
freeze = ap_run(['python', '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
(RUN_DIR / 'environment_latest.txt').write_text(freeze)


3275

## 3. Verify the GPU inside the prediction environment
This performs a small JAX calculation on the GPU, rather than checking only whether Colab reports a GPU.


In [ ]:
subprocess.run(['nvidia-smi'], check=True)
ap_run(['python', '-c', "import sys; import pkg_resources; import pdbfixer; import jax; import jax.numpy as jnp; import importlib.metadata as m; "
        "print('Python:', sys.executable); print('AlphaPulldown:', m.version('alphapulldown')); print('JAX:', jax.__version__); "
        "devices=jax.devices('gpu'); print(devices); "
        "x=jax.device_put(jnp.ones((32,32)), devices[0]); "
        "print('GPU calculation:', (x@x).block_until_ready()[0,0])"])


CompletedProcess(args=['/usr/local/bin/conda', 'run', '--no-capture-output', '-p', '/content/ap_env', 'python', '-c', "import sys; import pkg_resources; import pdbfixer; import jax; import jax.numpy as jnp; import importlib.metadata as m; print('Python:', sys.executable); print('AlphaPulldown:', m.version('alphapulldown')); print('JAX:', jax.__version__); devices=jax.devices('gpu'); print(devices); x=jax.device_put(jnp.ones((32,32)), devices[0]); print('GPU calculation:', (x@x).block_until_ready()[0,0])"], returncode=0)

## 4. Download AlphaFold2 model parameters
The December 2022 archive contains the current AF2 Multimer v3 parameter sets. Downloading newer AlphaPulldown does not require different AF2 weights. The archive is several GB; a fresh Colab runtime must download again. Interrupted downloads resume; existing parameter files are checked before reuse. Parameters remain on Colab's local disk to reduce Drive traffic.


In [ ]:
import tarfile
PARAM_ROOT.mkdir(parents=True, exist_ok=True)
param_dir = PARAM_ROOT / 'params'
param_dir.mkdir(exist_ok=True)
required_weights = [param_dir / f'params_{name}.npz' for name in MODEL_NAMES]
if not all(p.is_file() and p.stat().st_size > 0 for p in required_weights):
    archive = Path('/content/alphafold_params_2022-12-06.tar')
    url = 'https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar'
    subprocess.run(['wget', '-c', '-O', str(archive), url], check=True)
    # Extract regular parameter files only; reject paths outside the destination.
    with tarfile.open(archive) as tf:
        for member in tf:
            if not member.isfile():
                continue
            dest = (param_dir / member.name).resolve()
            if not dest.is_relative_to(param_dir.resolve()):
                raise RuntimeError(f'Unexpected archive path: {member.name}')
            if dest.suffix != '.npz':
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            with tf.extractfile(member) as src, dest.open('wb') as dst:
                shutil.copyfileobj(src, dst)
missing = [str(p) for p in required_weights if not p.is_file() or p.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Missing parameters: {missing}')
# Read compressed arrays to detect truncated/corrupt weights, once per runtime.
ap_run(['python', '-c', "import numpy as np, pathlib; "
        f"files={list(map(str, required_weights))!r}; "
        "[(print(pathlib.Path(f).name), [a[k].shape for k in a.files], a.close()) "
        "for f in files for a in [np.load(f, allow_pickle=False)]]; print('Weights verified')"])


CompletedProcess(args=['/usr/local/bin/conda', 'run', '--no-capture-output', '-p', '/content/ap_env', 'python', '-c', "import numpy as np, pathlib; files=['/content/par/params/params_model_1_multimer_v3.npz', '/content/par/params/params_model_2_multimer_v3.npz', '/content/par/params/params_model_3_multimer_v3.npz', '/content/par/params/params_model_4_multimer_v3.npz', '/content/par/params/params_model_5_multimer_v3.npz']; [(print(pathlib.Path(f).name), [a[k].shape for k in a.files], a.close()) for f in files for a in [np.load(f, allow_pickle=False)]]; print('Weights verified')"], returncode=0)

## 5. Validate features and save a stable job manifest
Only load pickles you generated or obtained from a trusted source. This cell checks every requested object in the installed environment, hashes features for provenance, and records pair order. The manifest supports whole proteins, including bait homodimers if a name occurs in both lists. It does not generate all-versus-all pairs.


In [16]:
import re, itertools, csv

def read_names(path):
    lines = [
        line.strip() for line in path.read_text().splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]

    if any(line.startswith(">") for line in lines):
        names = [
            line[1:].split()[0]
            for line in lines if line.startswith(">")
        ]
        # Also accept original UniProt headers: >sp|P02919|...
        names = [
            name.split("|")[1] if name.startswith(("sp|", "tr|")) else name
            for name in names
        ]
    else:
        names = lines

    if not names:
        raise ValueError(f"Empty protein list: {path}")
    if len(names) != len(set(names)):
        raise ValueError(f"Duplicate identifiers in {path}")
    for name in names:
        if not re.fullmatch(r"[A-Za-z0-9_.-]+", name):
            raise ValueError(f"Invalid protein identifier: {name!r}")
    return names

def digest_file(path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

baits, preys = read_names(BAIT_LIST), read_names(PREY_LIST)
feature_paths = {}
for name in dict.fromkeys(baits + preys):
    options = [FEATURE_DIR / (name + suffix) for suffix in ('.pkl', '.pkl.xz')]
    matches = [p for p in options if p.is_file()]
    if len(matches) != 1:
        raise ValueError(f'{name}: expected exactly one .pkl or .pkl.xz in {FEATURE_DIR}; found {matches}')
    feature_paths[name] = matches[0]

validator = Path('/content/validate_ap_features.py')
validator.write_text("""import json, sys, pickle, lzma
items=json.loads(sys.argv[1]); paired=sys.argv[2]=='True'
for name,path in items.items():
    opener=lzma.open if path.endswith('.xz') else open
    with opener(path,'rb') as handle: obj=pickle.load(handle)
    if not hasattr(obj,'feature_dict') or not hasattr(obj,'sequence'):
        raise TypeError(f'{name}: expected an AlphaPulldown MonomericObject, not raw features/MSA')
    if obj.description != name:
        raise ValueError(f'{name}: object description is {obj.description!r}; names must match')
    for key in ('aatype','residue_index','msa'):
        if key not in obj.feature_dict: raise ValueError(f'{name}: missing {key}')
    if paired and (getattr(obj,'skip_msa',False) or 'msa_all_seq' not in obj.feature_dict):
        raise ValueError(f'{name}: pairing features missing; regenerate features or intentionally set PAIR_MSA=False')
    print(f'{name}: {len(obj.sequence)} residues, MSA shape {obj.feature_dict["msa"].shape}')
""")
ap_run(['python', str(validator), json.dumps({k:str(v) for k,v in feature_paths.items()}), str(PAIR_MSA)])
settings = dict(alphapulldown=AP_VERSION, models=MODEL_NAMES, recycles=NUM_RECYCLES,
                predictions_per_model=PREDICTIONS_PER_MODEL, pair_msa=PAIR_MSA,
                skip_templates=SKIP_TEMPLATES, base_seed=BASE_SEED,
                baits=baits, preys=preys,
                feature_sha256={k:digest_file(v) for k,v in feature_paths.items()})
manifest_path = RUN_DIR / 'run_settings.json'
if manifest_path.exists() and json.loads(manifest_path.read_text()) != settings:
    raise RuntimeError('Run inputs/settings changed. Choose a new RUN_NAME to avoid mixing predictions.')
manifest_path.write_text(json.dumps(settings, indent=2))
fingerprint = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
jobs = []
for index, (bait, prey) in enumerate(itertools.product(baits, preys), 1):
    job_id = f'{index:05d}_' + hashlib.sha256(f'{bait}+{prey}'.encode()).hexdigest()[:12]
    jobs.append(dict(index=index, bait=bait, prey=prey, job_id=job_id))
with (RUN_DIR / 'jobs.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['index','bait','prey','job_id'])
    writer.writeheader(); writer.writerows(jobs)
print(f'{len(baits)} bait(s) × {len(preys)} prey(s) = {len(jobs)} jobs')
print(f'{len(MODEL_NAMES) * PREDICTIONS_PER_MODEL} predictions per job')
print('First jobs:', jobs[:5])


RuntimeError: Run inputs/settings changed. Choose a new RUN_NAME to avoid mixing predictions.

## 6. Run the selected jobs
Runs each pair with AlphaPulldown's current structure-prediction entry point. Each job has its own folder and log on Drive. Failed jobs stop the screen by default. Partial jobs use AlphaPulldown's native `allow_resume`; successful jobs get a completion marker only after the expected ranked structures and confidence entries exist. Do not change settings while resuming. Direct Drive output prioritises persistence but can be slower than local disk.


In [12]:
if not isinstance(START_JOB, int) or not 1 <= START_JOB <= len(jobs):
    raise ValueError('START_JOB must be within the manifest, starting at 1.')
last_job = len(jobs) if END_JOB is None else END_JOB
if not isinstance(last_job, int) or not START_JOB <= last_job <= len(jobs):
    raise ValueError('END_JOB must be None or within the manifest and >= START_JOB.')
expected_models = len(MODEL_NAMES) * PREDICTIONS_PER_MODEL
failures = []
for job in jobs[START_JOB - 1:last_job]:
    out = RUN_DIR / 'predictions' / job['job_id']
    out.mkdir(parents=True, exist_ok=True)
    marker = out / 'colab_complete.json'
    ranking_path = out / 'ranking_debug.json'
    def results_complete():
        try:
            ranking = json.loads(ranking_path.read_text())
            return (len(ranking['order']) == expected_models and
                    all((out / f'ranked_{i}.pdb').is_file() and
                        (out / f'ranked_{i}.pdb').stat().st_size > 0
                        for i in range(expected_models)))
        except (OSError, ValueError, KeyError, TypeError):
            return False
    if marker.exists() and json.loads(marker.read_text()).get('fingerprint') == fingerprint and results_complete():
        print(f"Skip completed {job['index']}: {job['bait']} + {job['prey']}")
        continue
    args = ['run_structure_prediction.py', f"--input={job['bait']}+{job['prey']}",
            f'--features_directory={FEATURE_DIR}', f'--data_directory={PARAM_ROOT}',
            f'--output_directory={out}', '--fold_backend=alphafold2',
            '--protein_delimiter=+', '--use_ap_style=False', '--allow_resume=True',
            '--models_to_relax=None', '--convert_to_modelcif=True',
            f'--num_cycle={NUM_RECYCLES}', f'--num_predictions_per_model={PREDICTIONS_PER_MODEL}',
            '--model_names=' + ','.join(MODEL_NAMES), f'--pair_msa={PAIR_MSA}',
            f'--skip_templates={SKIP_TEMPLATES}', f"--random_seed={BASE_SEED + job['index']}"]
    command = [CONDA, 'run', '--no-capture-output', '-p', str(ENV_PREFIX), *args]
    print(f"Run {job['index']}/{len(jobs)}: {job['bait']} + {job['prey']}", flush=True)
    started = time.time()
    with (out / 'prediction.log').open('a') as log:
        log.write('\nCommand: ' + repr(command) + '\n'); log.flush()
        process = subprocess.Popen(command, env=RUNTIME_ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=15)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if returncode != 0 or not results_complete():
        failures.append(job['index'])
        print('Failed or incomplete; inspect', out / 'prediction.log')
        if not CONTINUE_ON_ERROR:
            raise RuntimeError(f"Job {job['index']} failed; rerun after resolving its log error.")
        continue
    marker.write_text(json.dumps(dict(fingerprint=fingerprint, job=job,
                                      seconds=time.time()-started), indent=2))
print('Finished selected range. Failed jobs:', failures)


Run 1/656: P02919 + A5A616
I1004 13:26:59.868709 139134056830784 fold_preparation.py:100] Decompressing /content/drive/MyDrive/colab_ec/Escherichia_coli/features/P02919_feature_metadata_2025-06-16.json.xz to /content/drive/MyDrive/colab_ec/out/1b_K12/predictions/00001_b61e4313c884/P02919_feature_metadata_2025-06-16.json
I1004 13:27:00.508960 139134056830784 fold_preparation.py:100] Decompressing /content/drive/MyDrive/colab_ec/Escherichia_coli/features/A5A616_feature_metadata_2025-06-16.json.xz to /content/drive/MyDrive/colab_ec/out/1b_K12/predictions/00001_b61e4313c884/A5A616_feature_metadata_2025-06-16.json
I1004 13:27:08.971265 139134056830784 alphafold2_backend.py:693] Now running predictions on P02919_and_A5A616. Checking existing results...
I1004 13:27:08.972421 139134056830784 alphafold2_backend.py:734] 0 models are already completed.
I1004 13:27:08.972513 139134056830784 alphafold2_backend.py:798] Now running predictions on P02919_and_A5A616 using model_1_multimer_v3_pred_0
I10

KeyboardInterrupt: 

## 7. Export a confidence summary
Writes one row per ranked prediction, with ipTM/pTM when present in result pickles. The combined ranking score is read from `ranking_debug.json`; it is not ipTM alone. This summary is for prioritising candidates, not demonstrating interaction experimentally. Interface PAE and contact inspection remain necessary. Result pickles are retained for downstream analysis.


In [ ]:
summary_script = Path('/content/summarize_ap.py')
summary_script.write_text("""import pathlib, json, pickle, gzip, csv, sys
root=pathlib.Path(sys.argv[1]); rows=[]
with (root/'jobs.csv').open() as f: jobs=list(csv.DictReader(f))
for job in jobs:
    out=root/'predictions'/job['job_id']; path=out/'ranking_debug.json'
    if not path.exists(): continue
    ranking=json.loads(path.read_text()); scores=ranking.get('iptm+ptm',ranking.get('plddts',{}))
    for rank,model in enumerate(ranking.get('order',[])):
        row=dict(job_index=job['index'],bait=job['bait'],prey=job['prey'],rank=rank,
                 model=model,ranking_score=scores.get(model,''),iptm='',ptm='',
                 complete=(out/'colab_complete.json').exists(),output_dir=str(out))
        for suffix,opener in [('.pkl',open),('.pkl.gz',gzip.open)]:
            result=out/('result_'+model+suffix)
            if result.exists():
                with opener(result,'rb') as f: data=pickle.load(f)
                for key in ('iptm','ptm'):
                    if key in data: row[key]=float(data[key])
                break
        rows.append(row)
fields=['job_index','bait','prey','rank','model','ranking_score','iptm','ptm','complete','output_dir']
with (root/'confidence_summary.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
print(f'Wrote {len(rows)} rows to {root / "confidence_summary.csv"}')
""")
ap_run(['python', str(summary_script), str(RUN_DIR)])


## Troubleshooting and references
- **No GPU in JAX:** select a GPU runtime and rerun the environment check. Save the complete error; do not independently upgrade JAX over the pinned release.
- **Missing bait list:** create `baits.txt` on Drive or edit `BAIT_LIST`. Candidate names default to your original `syd.txt`.
- **Feature loading/pairing errors:** use AlphaPulldown monomer objects compatible with 2.9.1. Raw `.a3m` files cannot be passed here. Disabling pairing changes the modelling protocol; regenerate suitable features when pairing is intended.
- **Out of memory:** try a smaller complex or a runtime with more GPU memory. Reducing recycles alone does not reliably solve memory limits.
- **Interrupted session:** restart setup, use the same run name and unchanged inputs, and set the desired job range. A partially written model may need troubleshooting; native resume is not a guarantee against all interrupted writes.
- **Changed protocol:** choose a new run name. Do not combine scores from different settings without recording those differences.

References: [AlphaPulldown 2.9.1](https://pypi.org/project/alphapulldown/2.9.1/), [AF2 dependencies](https://github.com/KosinskiLab/AlphaPulldown/blob/main/pyproject.toml), [prediction CLI](https://github.com/KosinskiLab/AlphaPulldown/blob/main/alphapulldown/scripts/run_structure_prediction.py), [CondaColab](https://github.com/conda-incubator/condacolab).


## Extra: Generate features for a protein of interest

In [13]:
FASTA_PATH = Path("/content/drive/MyDrive/colab_ec/mrcB_799-844.fasta")

ap_run([
    "create_individual_features.py",
    "--data_pipeline=alphafold2",
    f"--fasta_paths={FASTA_PATH}",
    f"--output_dir={FEATURE_DIR}",
    "--use_mmseqs2=True",
    "--re_search_templates_mmseqs2=False",
    "--max_template_date=2026-10-04",
    "--skip_existing=True",
    "--compress_features=False",
])

CompletedProcess(args=['/usr/local/bin/conda', 'run', '--no-capture-output', '-p', '/content/ap_env', 'create_individual_features.py', '--data_pipeline=alphafold2', '--fasta_paths=/content/drive/MyDrive/colab_ec/mrcB_799-844.fasta', '--output_dir=/content/drive/MyDrive/colab_ec/Escherichia_coli/features', '--use_mmseqs2=True', '--re_search_templates_mmseqs2=False', '--max_template_date=2026-10-04', '--skip_existing=True', '--compress_features=False'], returncode=0)

In [14]:
names = read_names(FASTA_PATH)
for name in names:
    path = FEATURE_DIR / f"{name}.pkl"
    print(path, "—", f"{path.stat().st_size:,} bytes" if path.exists() else "MISSING")

/content/drive/MyDrive/colab_ec/Escherichia_coli/features/mrcB_799-844.pkl — 88,364 bytes
